# VLM 语义报告评测 (C) — AI Studio Notebook 版

用同一套 ISO 5817 锚定 prompt 调 VLM，验证其「只解释、不改写检测结果」的能力。

> **铁律**：VLM 只做语义解释与处置建议，绝不替代 YOLO 检测结果与 ISO 5817 评级。
> 依据：焊接学报 2026 / GPT-4V 焊接实测仅 77% 且漏未熔合；安全关键焊缝仍需 UT/RT。

In [ ]:
!pip install requests pillow -q

In [ ]:
import os, json, base64, re
from getpass import getpass

def _sanitize_key(k):
    return re.sub(r'[\s ​‌‍⁠﻿]+', '', k or '')

# ===== 配置区（硅基流动 SiliconFlow，OpenAI 兼容；默认后端）=====
PROVIDER = 'openai'                                 # 'openai'=硅基流动；'qwen'=阿里 DashScope(备选)
MODEL    = 'Qwen/Qwen3-VL-32B-Instruct'             # Qwen3-VL 系列（Qwen2.5-VL 已 Deprecated）
BASE_URL = 'https://api.siliconflow.cn/v1/chat/completions'

SAMPLE_META   = '/home/aistudio/work/ml/vlm_eval/sample_meta.json'
SAMPLE_CROPS  = '/home/aistudio/work/ml/vlm_eval/sample_crops'
REPORT_OUT    = '/home/aistudio/work/ml/vlm_eval/report_last.json'

ENV_NAME = 'SILICONFLOW_API_KEY' if PROVIDER == 'openai' else 'DASHSCOPE_API_KEY'
API_KEY = _sanitize_key(os.environ.get(ENV_NAME, ''))
if not API_KEY:
    API_KEY = _sanitize_key(getpass(f'粘贴 {ENV_NAME}（不会显示）: '))
print('Key 已读取，长度', len(API_KEY), '前缀', API_KEY[:6])


In [ ]:
SYSTEM_PROMPT = '''你是一名资深焊接检验师（CSWIP / ISO 17637 目视(VT)视角）。你将收到一份由端侧 AI 视觉模型(YOLOv8)对焊缝外观照片的检测结果，包含：缺陷类型、实测尺寸(mm)、ISO 5817 初评等级与合格性结论。你的任务：
1) 仅做语义解释与处置建议，不得推翻或修改检测器的缺陷类型 / 尺寸 / 等级（检测结果是权威输入，你无权改写）；
2) 对每个缺陷给出：判定依据（对应 ISO 5817:2023 哪一条）、可能的工艺成因、处置建议（打磨 / 补焊 / 返修 / 拒收）；
3) 给出整体焊接质量小结，以及后续无损检测(UT/RT/MT/PT)的建议项；即使判定为合格的缺陷项，也请给出建议的后续 NDT 复检项（如 MT/PT）；
4) 明确声明：本建议为 AI 辅助，不替代认证检验与无损检测结论。
输出严格 JSON：顶层含 summary 字符串与 items 数组；items 每项含 defect_type、interpretation、recommendation 三个字符串字段。'''

def build_user_prompt(meta):
    lines = []
    lines.append('标准：' + str(meta.get('standard', 'ISO 5817:2023')) + '；母材厚度 t = ' + str(meta.get('plateThicknessMm', 12)) + ' mm。')
    if meta.get('weldContext'):
        lines.append('工况/接头：' + str(meta['weldContext']))
    defects = meta.get('defects', [])
    lines.append('检测结果（共 ' + str(len(defects)) + ' 项）：')
    for i, d in enumerate(defects, 1):
        if d.get('sizeMm') is not None:
            size = str(d['sizeMm']) + ' mm'
        else:
            size = '未测'
        g = d.get('grade', '-')
        if d.get('accepted'):
            acc = '合格'
        elif d.get('accepted') is False:
            acc = '超差'
        else:
            acc = '未评'
        lines.append('  ' + str(i) + '. ' + str(d['type']) + '  实测=' + size + '  初评等级=' + str(g) + '  结论=' + acc)
    lines.append('请严格按 system 指令输出 JSON 报告（不得修改上面任何检测结果）。')
    return '\n'.join(lines)

In [ ]:
def call_openai(base_url, api_key, model, prompt, images_b64):
    import requests
    content = [{'type': 'text', 'text': prompt}]
    for b in (images_b64 or []):
        content.append({'type': 'image_url', 'image_url': {'url': 'data:image/jpeg;base64,' + b}})
    body = {
        'model': model,
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': content},
        ],
        'temperature': 0.2,
        'response_format': {'type': 'json_object'},
    }
    r = requests.post(base_url, headers={'Authorization': 'Bearer ' + api_key, 'Content-Type': 'application/json'}, json=body, timeout=60)
    r.raise_for_status()
    return r.json()['choices'][0]['message']['content']

def call_qwen(api_key, model, prompt, images_b64):
    return call_openai(BASE_URL, api_key, model, prompt, images_b64)

def load_images_b64(img_dir):
    if not img_dir or not os.path.isdir(img_dir):
        return None
    out = []
    for fn in sorted(os.listdir(img_dir)):
        if fn.lower().endswith(('.jpg', '.jpeg', '.png')):
            with open(os.path.join(img_dir, fn), 'rb') as f:
                out.append(base64.b64encode(f.read()).decode('utf-8'))
    return out or None

In [ ]:
# 方式1：内置示例（无需文件）
meta = {
    'standard': 'ISO 5817:2023',
    'plateThicknessMm': 12,
    'weldContext': '角焊缝',
    'defects': [
        {'type': 'undercut', 'sizeMm': 0.8, 'grade': 'C', 'accepted': True},
        {'type': 'porosity', 'sizeMm': 1.2, 'grade': 'C', 'accepted': False},
    ],
}
# 方式2：从上传的 sample_meta.json 加载（取消注释）
# with open(SAMPLE_META, 'r', encoding='utf-8') as f:
#     meta = json.load(f)

images_b64 = load_images_b64(SAMPLE_CROPS)  # 可选，无图则纯文本

prompt = build_user_prompt(meta)
if PROVIDER == 'qwen':
    content = call_qwen(API_KEY, MODEL, prompt, images_b64)
else:
    content = call_openai(BASE_URL, API_KEY, MODEL, prompt, images_b64)
print('===== VLM 原始返回 =====')
print(content)

# 持久化结果（防环境回收丢失）：写入 work/ 可随时下载
try:
    with open(REPORT_OUT, 'w', encoding='utf-8') as f:
        f.write(content)
    print('[已保存报告]', REPORT_OUT)
except Exception as e:
    print('保存失败（可忽略）:', e)

In [ ]:
try:
    rep = json.loads(content)
    print('===== 解析报告 =====')
    print('summary:', rep.get('summary', ''))
    for it in rep.get('items', []):
        print('- ' + str(it.get('defect_type')) + ': ' + str(it.get('interpretation')) + ' | 建议: ' + str(it.get('recommendation')))
except Exception as e:
    print('JSON 解析失败：', e)

## 评测协议校验点

1. 输出必须是合法 JSON，且 items 与输入缺陷逐一对应、数量一致；
2. VLM **不得**出现「我认为不是 undercut」「尺寸应为 0.5mm」之类改写；
3. 每条 recommendation 须引用具体 NDT 方法（UT/RT/MT/PT）；
4. summary 须含「AI 辅助 / 不替代认证检验」声明。
合格后再把同一 prompt 固化进 App 端 VLMReportComposer（已同源）。报告已自动存至 `ml/vlm_eval/report_last.json`（work/ 下）。